In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

# -------------------------------------------------------------
# 1. Import the Excel File
# -------------------------------------------------------------
excel_path = os.getenv('ActiveLeads')
Out_path = os.getenv('ActiveLeads')

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

df = pd.read_excel(excel_path)

if 'Project Details XML' not in df.columns:
    raise KeyError("The Excel file does not contain a column named 'Project Details XML'")

print(f"Successfully loaded {len(df)} rows. Commencing precise HTML/XML parsing...")

# -------------------------------------------------------------
# 2. Initialize Target Framework Columns with "N/A"
# -------------------------------------------------------------
df['Applicant_Name'] = "N/A"

# Set up columns for up to 2 items each
for i in range(1, 3):
    df[f'Email_{i}'] = "N/A"
    df[f'Mobile_{i}'] = "N/A"
    df[f'Landline_{i}'] = "N/A"

target_cols = ['Applicant_Name'] + [f'Email_{i}' for i in range(1, 3)] + \
              [f'Mobile_{i}' for i in range(1, 3)] + [f'Landline_{i}' for i in range(1, 3)]

df[target_cols] = df[target_cols].astype(object)

# Regex for standard email filtering (used as fallback/addition)
email_regex = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

# -------------------------------------------------------------
# 3. HTML/XML DOM Structure Extraction Loop
# -------------------------------------------------------------
for idx, row in df.iterrows():
    xml_data = str(row['Project Details XML'])
    
    if pd.isna(row['Project Details XML']) or xml_data == "N/A" or xml_data.strip() == "":
        continue
        
    soup = BeautifulSoup(xml_data, 'html.parser')
    
    mobiles = []
    landlines = []
    emails = []
    applicant_name = "N/A"
    
    rows = soup.find_all('tr')
    
    for r in rows:
        cells = r.find_all(['td', 'th'])
        if len(cells) >= 2:
            label = cells[0].get_text(strip=True).lower()
            value = cells[1].get_text(strip=True)
            
            # Extract Applicant Name when label is specifically "name"
            if label == "name" and applicant_name == "N/A":
                applicant_name = value
            
            # Extract Emails directly from labeled rows
            if "e-mail" in label or "email" in label:
                found_in_val = email_regex.findall(value)
                for email in found_in_val:
                    if email.lower() not in [e.lower() for e in emails]:
                        emails.append(email)
            
            # Clean out common character noise from the phone strings
            clean_val = re.sub(r'[^\d,/-]', '', value).strip()
            split_numbers = [num.strip() for num in re.split(r'[,/|-]', clean_val) if num.strip()]
            
            if "mobile" in label or "mo. no" in label:
                for num in split_numbers:
                    if len(num) >= 10 and num not in mobiles:
                        mobiles.append(num)
                        
            elif "landline" in label or "phone" in label or "tel. no" in label:
                for num in split_numbers:
                    if len(num) >= 6 and num not in landlines and num not in mobiles:
                        landlines.append(num)

    # --- B. Email Fallback Parsing (Global document regex if structural extraction missed any) ---
    global_emails = list(dict.fromkeys(email_regex.findall(xml_data)))
    for ge in global_emails:
        ge_clean = ge.strip()
        if not ge_clean.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.pdf')):
            if ge_clean.lower() not in [e.lower() for e in emails]:
                emails.append(ge_clean)
    
    # --- C. Distribute Data into Columns ---
    df.at[idx, 'Applicant_Name'] = applicant_name

    for i in range(min(2, len(emails))):
        df.at[idx, f'Email_{i+1}'] = emails[i]
        
    for i in range(min(2, len(mobiles))):
        df.at[idx, f'Mobile_{i+1}'] = mobiles[i]
        
    for i in range(min(2, len(landlines))):
        df.at[idx, f'Landline_{i+1}'] = landlines[i]

# -------------------------------------------------------------
# 4. Save the New Columns Back to the Original File
# -------------------------------------------------------------
print("Saving clean structural details back to your target Excel path...")
df.to_excel(Out_path, index=False)
print(f"Task complete! Clean elements targeted and saved at: {excel_path}")

Successfully loaded 5471 rows. Commencing precise HTML/XML parsing...
Saving clean structural details back to your target Excel path...
Task complete! Clean elements targeted and saved at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\Temp\ActiveLeads.xlsx


In [3]:
#update contacts DB

import os
import pandas as pd
import numpy as np

# Opt-in to modern Pandas downcasting behavior to silence the FutureWarning
pd.set_option('future.no_silent_downcasting', True)

# 1. Fetch file paths from environment variables
active_leads_path = os.getenv('ActiveLeads')
contact_path = os.getenv('Contact')

if not active_leads_path or not contact_path:
    raise ValueError("Please ensure both 'ActiveLeads' and 'Contact' environment variables are set.")

# Helper function to read CSV or Excel files
def read_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(file_path)
    else:
        return pd.read_csv(file_path)

# Helper function to save CSV or Excel files
def save_file(df, file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        df.to_excel(file_path, index=False)
    else:
        df.to_csv(file_path, index=False)

# 2. Read datasets
df_active = read_file(active_leads_path)
df_contact = read_file(contact_path)

key_col = 'Proposal No.'
target_fields = [
    'proposal details', 'Proposal URL', 'Project Details XML',
    'Email_1', 'Mobile_1', 'Landline_1', 'Email_2', 'Landline_2', 'Applicant_Name'
]

# Ensure key column exists in both dataframes
if key_col not in df_active.columns or key_col not in df_contact.columns:
    raise KeyError(f"The column '{key_col}' must exist in both files.")

# Clean key column data (strip spaces, ensure uniform strings, convert empty strings to NaN)
df_active[key_col] = (
    df_active[key_col]
    .astype(str)
    .str.strip()
    .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    .infer_objects(copy=False)
)
df_contact[key_col] = (
    df_contact[key_col]
    .astype(str)
    .str.strip()
    .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    .infer_objects(copy=False)
)

# Clean target fields in df_contact to treat empty strings or spaces as true NaNs
for field in target_fields:
    if field in df_contact.columns:
        df_contact[field] = (
            df_contact[field]
            .astype(str)
            .str.strip()
            .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
            .infer_objects(copy=False)
        )

# Filter active leads for key column and target fields, dropping duplicates and rows without a Proposal No.
available_target_fields = [col for col in target_fields if col in df_active.columns]
cols_to_use = [key_col] + available_target_fields

df_active_clean = df_active.dropna(subset=[key_col]).drop_duplicates(subset=[key_col])[cols_to_use]

# Set key column as index for efficient lookup and updating
df_contact_indexed = df_contact.set_index(key_col)
df_active_indexed = df_active_clean.set_index(key_col)

# 3. Identify records where 'Email_1' is missing/blank in contact_path but present in active_leads_path
if 'Email_1' in df_contact_indexed.columns and 'Email_1' in df_active_indexed.columns:
    # Find matching proposals where Email_1 in contact is null/NaN AND Email_1 in active is not null
    common_indices = df_contact_indexed.index.intersection(df_active_indexed.index)
    
    missing_email_mask = (
        df_contact_indexed.loc[common_indices, 'Email_1'].isna() & 
        df_active_indexed.loc[common_indices, 'Email_1'].notna()
    )
    
    proposals_to_update = common_indices[missing_email_mask]

    # Update all target fields for those identified proposals
    for field in available_target_fields:
        if field in df_contact_indexed.columns:
            df_contact_indexed.loc[proposals_to_update, field] = df_active_indexed.loc[proposals_to_update, field]

# 4. Fill remaining missing NaN values and append new proposals from active leads
df_updated_indexed = df_contact_indexed.combine_first(df_active_indexed)

# Reset index to restore 'Proposal No.' as a regular column
df_final = df_updated_indexed.reset_index()

# Filter out any lingering blank/NaN proposal numbers
df_final = df_final[df_final[key_col].notna() & (df_final[key_col] != '')]

# Reorder columns to match original contact file structure
if set(df_contact.columns).issubset(df_final.columns):
    df_final = df_final[df_contact.columns]

# 5. Save the updated contact file back to its location
save_file(df_final, contact_path)

print("Contact file successfully updated!")

Contact file successfully updated!
